# Exercício: Regras de Associação

Objetivos: Exercitar os conceitos referente à regras de associação.

## Questão 1

Utilizando os dados referente a postagens no Telegram, descubra regras que associem
entidades nomeadas presentes nessas mensagens.

Para cada mensagem identifique uma lista de entidades nomeadas. Utilizando essas listas
de entidades nomeadas descubra regras de associação interessantes.

a. O algoritmo de classificação: Apriori, FP-Growth (Frequent Pattern Growth) e ECLAT (Equivalence Class Clustering and bottom-up Lattice Traversal);

In [1]:
import duckdb
import spacy
from tqdm import tqdm
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from collections import Counter
from langdetect import detect, LangDetectException

from tqdm.auto import tqdm

In [2]:
conn = duckdb.connect()

telegram_data = conn.read_csv("../data/fakeTelegram.BR_2022.csv")

query = """
    SELECT * FROM telegram_data
"""

df = conn.execute(query).fetchdf()

df.head()

,date_message,id_member_anonymous,id_group_anonymous,media,media_type,media_url,has_media,has_media_url,trava_zap,text_content_anonymous,dataset_info_id,date_system,score_sentiment,score_misinformation,id_message,message_type,messenger,media_name,media_md5
0,2022-10-05 06:25:04,1078cc958f0febe28f4d03207660715f,12283e08a2eb5789201e105b34489ee7,None,None,None,False,False,False,Então é Fato Renato o áudio que eu ouvi no wha...,5,2022-10-05 06:25:28.863641,0.0000,NaN,16385,Texto,telegram,None,None
1,2022-10-05 06:25:08,None,12283e08a2eb5789201e105b34489ee7,None,None,None,False,False,False,"Saiu no YouTube do presidente a 8 horas atrás,...",5,2022-10-05 06:25:28.926311,0.0644,NaN,16386,Texto,telegram,None,None
2,2022-10-05 06:26:28,92a2d8fd7144074f659d1d29dc3751da,9f2d7394334eb224c061c9740b5748fc,None,None,None,False,False,False,"É isso, nossa parte já foi quase toda feita. N...",5,2022-10-05 06:26:29.361949,-0.3551,0.157242,16366,Texto,telegram,None,None
3,2022-10-05 06:27:28,d60aa38f62b4977426b70944af4aff72,c8f2de56550ed0bf85249608b7ead93d,94dca4cda503100ebfda7ce2bcc060eb.jpg,image/jpg,None,True,False,False,GENTE ACHEI ELES EM UMA SEITA MAÇONÁRICA,5,2022-10-05 06:27:29.935624,0.0000,NaN,19281,Imagem,telegram,None,94dca4cda503100ebfda7ce2bcc060eb
4,2022-10-05 06:27:44,cd6979b0b5265f08468fa1689b6300ce,e56ec342fc599ebb4ed89655eb6f03aa,5ad5c8bbe9da93a37fecf3e5aa5b0637.jpg,image/jpg,None,True,False,False,None,5,2022-10-05 06:28:29.316325,NaN,NaN,507185,Imagem,telegram,None,5ad5c8bbe9da93a37fecf3e5aa5b0637


In [3]:
df.shape

(557586, 19)

In [4]:
df = conn.execute(f"""
    SELECT * 
    FROM telegram_data
    WHERE trava_zap IS NOT TRUE
""").fetch_df()

In [5]:
df.shape

(557570, 19)

In [6]:
df_ = conn.execute("SELECT DISTINCT * FROM df").fetch_df()

In [7]:
query = """
SELECT *
FROM df_
WHERE array_length(string_split(text_content_anonymous, ' ')) >= 5
"""

df = conn.execute(query).fetch_df()

df.shape

(336944, 19)

In [8]:
# !pip install spacy
# !pip install mlxtend
# !python -m spacy download pt_core_news_lg

In [ ]:
tqdm.pandas()

def eclat(df, min_support):
    transactions = []
    for index, row in df.iterrows():
        transaction = set(row[row == 1].index)
        transactions.append(transaction)
    items = {}
    for i, transaction in enumerate(transactions):
        for item in transaction:
            if item not in items:
                items[item] = set()
            items[item].add(i)

    frequent_itemsets = []
    k = 1
    
    current_itemsets = []
    for item, tids in items.items():
        support = len(tids) / len(transactions)
        if support >= min_support:
            frequent_itemsets.append((frozenset([item]), support))
            current_itemsets.append(frozenset([item]))
            
    k = 2
    while current_itemsets:
        next_gen_itemsets = set()
        for i in range(len(current_itemsets)):
            for j in range(i + 1, len(current_itemsets)):
                itemset1 = current_itemsets[i]
                itemset2 = current_itemsets[j]
                
                candidate = itemset1.union(itemset2)
                if len(candidate) == k:
                    tids1 = items[next(iter(itemset1))]
                    for item in itemset1:
                        tids1 = tids1.intersection(items[item])

                    tids2 = items[next(iter(itemset2))]
                    for item in itemset2:
                        tids2 = tids2.intersection(items[item])
                    
                    candidate_tids = tids1.intersection(tids2)
                    support = len(candidate_tids) / len(transactions)
                    
                    if support >= min_support:
                        frequent_itemsets.append((candidate, support))
                        next_gen_itemsets.add(candidate)
                        
        current_itemsets = list(next_gen_itemsets)
        k += 1

    return frequent_itemsets


# carregar o modelo de linguagem
print("Carregando o modelo de linguagem spaCy 'pt_core_news_lg'...")
nlp = spacy.load('pt_core_news_lg')

# filtrar mensagens para manter apenas o português
def is_portuguese(text):
    if not isinstance(text, str) or len(text.strip()) < 20:
        return False
    try:
        return detect(text) == 'pt'
    except LangDetectException:
        return False

print("\nIniciando a filtragem de idioma (com barra de progresso)...")
df['is_pt'] = df['text_content_anonymous'].progress_apply(is_portuguese)
df_pt = df[df['is_pt']].copy()

print(f"\nDataset original: {len(df)} mensagens.")
print(f"Dataset em português: {len(df_pt)} mensagens.")


# extrair entidades nomeadas
print("\nIniciando o reconhecimento de entidades nomeadas (NER)...")
texts_to_process = df_pt['text_content_anonymous'].astype(str).tolist()
all_entities = []

for doc in tqdm(nlp.pipe(texts_to_process, disable=["parser"]), total=len(texts_to_process)):
    # extrai entidades (Pessoa, Organização, Local) e remove espaços extras.
    entities = [ent.text.strip() for ent in doc.ents if ent.label_ in ['PER', 'ORG', 'LOC']]
    if entities:
        all_entities.append(list(set(entities)))

df_pt_filtered = df_pt.iloc[:len(all_entities)].copy()
df_pt_filtered['entities'] = all_entities

# manter apenas mensagens com pelo menos 2 entidades para mineração
df_pt_filtered = df_pt_filtered[df_pt_filtered['entities'].apply(len) >= 2]

# preparar dados para os algoritmos de associação
all_entities_list = [entity for sublist in df_pt_filtered['entities'] for entity in sublist]
entity_counts = Counter(all_entities_list)

MINIMUM_FREQUENCY = 20 

frequent_entities = {entity for entity, count in entity_counts.items() if count >= MINIMUM_FREQUENCY}

print(f"\nNúmero de entidades únicas antes do filtro de frequência: {len(entity_counts)}")
print(f"Número de entidades após filtrar (frequência >= {MINIMUM_FREQUENCY}): {len(frequent_entities)}")

df_pt_filtered['entities_filtered'] = df_pt_filtered['entities'].apply(
    lambda entity_list: [entity for entity in entity_list if entity in frequent_entities]
)

# criar a lista de transações final
transactions = df_pt_filtered[df_pt_filtered['entities_filtered'].apply(len) >= 2]['entities_filtered'].tolist()

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transacional = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Formato do DataFrame transacional: {df_transacional.shape}")

# aplicar os algoritmos e gerar regras
if not df_transacional.empty:
    min_support_value = 0.01 

    print("\nExecutando algoritmos de mineração de regras de associação...")
    # FP-Growth
    frequent_itemsets_fpgrowth = fpgrowth(df_transacional, min_support=min_support_value, use_colnames=True)
    
    # Apriori
    frequent_itemsets_apriori = apriori(df_transacional, min_support=min_support_value, use_colnames=True)

    # ECLAT
    frequent_itemsets_eclat_result = eclat(df_transacional, min_support=min_support_value)
    
    print("\n--- Resultados dos Algoritmos ---")
    print(f"FP-Growth encontrou: {len(frequent_itemsets_fpgrowth)} conjuntos de itens frequentes")
    print(f"Apriori encontrou: {len(frequent_itemsets_apriori)} conjuntos de itens frequentes")
    print("ECLAT encontrou:", len(frequent_itemsets_eclat_result), "conjuntos frequentes")

    # gerar regras a partir do resultado do FP-Growth
    rules = association_rules(frequent_itemsets_fpgrowth, metric="lift", min_threshold=1.2)
    
    # ordenar por 'lift' e 'confidence' para ver as regras mais fortes
    rules_sorted = rules.sort_values(by=['lift', 'confidence'], ascending=False)
    
    print("\n--- REGRAS DE ASSOCIAÇÃO MAIS FORTES (LIFT > 1.2) ---")
    print(rules_sorted[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(20))
else:
    print("\nNenhuma transação válida encontrada após a filtragem. Tente ajustar os parâmetros `MINIMUM_FREQUENCY` ou `min_support_value`.")

Carregando o modelo de linguagem spaCy 'pt_core_news_lg'...

Iniciando a filtragem de idioma (com barra de progresso)...


  0%|          | 0/336944 [00:00<?, ?it/s]


Dataset original: 336944 mensagens.
Dataset em português: 273236 mensagens.

Iniciando o reconhecimento de entidades nomeadas (NER)...


  0%|          | 0/273236 [00:00<?, ?it/s]


Número de entidades únicas antes do filtro de frequência: 76190
Número de entidades após filtrar (frequência >= 20): 3703
Formato do DataFrame transacional: (93177, 3702)

Executando algoritmos de mineração de regras de associação...

--- Resultados dos Algoritmos ---
FP-Growth encontrou: 65 conjuntos de itens frequentes
Apriori encontrou: 65 conjuntos de itens frequentes

--- REGRAS DE ASSOCIAÇÃO MAIS FORTES (LIFT > 1.2) ---
           antecedents         consequents   support  confidence       lift
49           (Oficial)    (Canal, Verdade)  0.016098    0.998004  61.829134
44    (Canal, Verdade)           (Oficial)  0.016098    0.997340  61.829134
45    (Canal, Oficial)           (Verdade)  0.016098    1.000000  59.843931
48           (Verdade)    (Canal, Oficial)  0.016098    0.963391  59.843931
41           (Oficial)           (Verdade)  0.016098    0.998004  59.724482
40           (Verdade)           (Oficial)  0.016098    0.963391  59.724482
26        (USER, João)             (G